In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
## setting working directory
import os
os.chdir('/PHShome/ll1009/linke/chip_gwas_rev/')

## onek1k cell subsampling

In [ ]:
### reading in filtered data and prepare for subsampling
adata = sc.read_h5ad("Data/sc_annot_files/onek1k_sc_filtered.h5ad")
print("Full OneK1K:", adata.shape)

n_cell  = 50000
n_reps  = 20
out_path = "Data/sc_annot_files_subsets/onek1k"
cov_path = "Data/sc_annot_files_subsets/onek1k"

In [ ]:
for i_rep in range(n_reps):
    np.random.seed(i_rep)
    idx = np.random.choice(adata.shape[0], size=n_cell, replace=False)
    tmp = adata[idx].copy()

    # Save h5ad
    out_file = f"{out_path}/onek1k.ncell_50k_rep{i_rep}.h5ad"
    tmp.write(out_file)

    # Save covariate file — pool dummies from this subsample only
    df_cov = pd.DataFrame(index=tmp.obs.index)
    df_cov['const']   = 1
    df_cov['n_genes'] = tmp.obs['nFeature_RNA']

    pools = sorted(tmp.obs['pool_number'].unique())
    for pool in pools[1:]:
        df_cov[f'pool_{pool}'] = (tmp.obs['pool_number'] == pool).astype(int)

    cov_file = f"{cov_path}/onek1k.ncell_50k_rep{i_rep}.cov"
    df_cov.to_csv(cov_file, sep="\t")

    print(f"Rep {i_rep}: cells={tmp.shape[0]} pools={len(pools)} cov_cols={df_cov.shape[1]}")
    del tmp

print("Done.")

In [ ]:
### generate the dictionary script for looping
# Manifest for cell subsamples (2 columns: h5ad, cov)
with open("Data/sc_annot_files_subsets/helper_files/onek1k_cell_subsample_manifest.txt", "w") as f:
    for i_rep in range(n_reps):
        h5ad = f"/data/cteu/2_users/zyu_lab/linke/chip_gwas_rev/{out_path}/onek1k.ncell_50k_rep{i_rep}.h5ad"
        cov  = f"/data/cteu/2_users/zyu_lab/linke/chip_gwas_rev/{cov_path}/onek1k.ncell_50k_rep{i_rep}.cov"
        f.write(f"{h5ad}\t{cov}\tpredicted.celltype.l2\n")

print("Manifests written.")

## BM subsampling

In [3]:
### reading in filtered data and prepare for subsampling
adata = sc.read_h5ad("Data/sc_annot_files/BM_standard_design.h5ad")
print("Full BM standard design:", adata.shape)

n_cell  = 50000
n_reps  = 20
out_path = "Data/sc_annot_files_subsets/BM"
cov_path = "Data/sc_annot_files_subsets/BM"

Full BM standard design: (266271, 26911)


In [4]:
# Check cell type distribution
ct_counts = adata.obs['anno'].value_counts()
print("\nCell type counts (full dataset):")
print(ct_counts)

# Estimate what 50K subsample would look like
subsample_frac = 50000 / adata.shape[0]
print(f"\nSubsample fraction: {subsample_frac:.1%}")
print("\nExpected cells per type at 50K subsample:")
print((ct_counts * subsample_frac).astype(int))


Cell type counts (full dataset):
anno
CD4+ naive T cells           44009
Cytotoxic T cells            39450
CD14+ monocytes              37570
T helper cells               24797
Naive B cells                23515
CD8+ naive T cells           20877
NK cells                     19528
Erythroid cells               8954
Memory B cells                8822
Pre-B cells                   7523
cDCs                          4703
HSCs                          4323
CD16+ monocytes               3807
Neutrophil progenitors        3571
Pro-B cells                   3313
pDCs                          2790
Plasma cells                  2605
Erythroid progenitors         2562
ANK1-low erythroid cells      2552
Megakaryocyte progenitors      719
MSCs                           281
Name: count, dtype: int64

Subsample fraction: 18.8%

Expected cells per type at 50K subsample:
anno
CD4+ naive T cells           8263
Cytotoxic T cells            7407
CD14+ monocytes              7054
T helper cells         

In [6]:
for i_rep in range(n_reps):
    np.random.seed(i_rep)
    idx = np.random.choice(adata.shape[0], size=n_cell, replace=False)
    tmp = adata[idx].copy()

    h5ad_out = f"{out_path}/bm_standard.ncell_50k_rep{i_rep}.h5ad"
    tmp.write(h5ad_out)

    # Covariate — channel dummies from this subsample
    df_cov = pd.DataFrame(index=tmp.obs.index)
    df_cov['const']   = 1
    df_cov['n_genes'] = tmp.obs['n_genes']

    channels = sorted(tmp.obs['Channel'].unique())
    for ch in channels[1:]:
        df_cov[f'channel_{ch}'] = (tmp.obs['Channel'] == ch).astype(int)

    cov_out = f"{cov_path}/bm_standard.ncell_50k_rep{i_rep}.cov"
    df_cov.to_csv(cov_out, sep="\t", index=True)

    print(f"Rep {i_rep:2d}: cells={tmp.shape[0]:,} "
          f"channels={len(channels)} cov_cols={df_cov.shape[1]}")
    del tmp

Rep  0: cells=50,000 channels=63 cov_cols=64
Rep  1: cells=50,000 channels=63 cov_cols=64
Rep  2: cells=50,000 channels=63 cov_cols=64
Rep  3: cells=50,000 channels=63 cov_cols=64
Rep  4: cells=50,000 channels=63 cov_cols=64
Rep  5: cells=50,000 channels=63 cov_cols=64
Rep  6: cells=50,000 channels=63 cov_cols=64
Rep  7: cells=50,000 channels=63 cov_cols=64
Rep  8: cells=50,000 channels=63 cov_cols=64
Rep  9: cells=50,000 channels=63 cov_cols=64
Rep 10: cells=50,000 channels=63 cov_cols=64
Rep 11: cells=50,000 channels=63 cov_cols=64
Rep 12: cells=50,000 channels=63 cov_cols=64
Rep 13: cells=50,000 channels=63 cov_cols=64
Rep 14: cells=50,000 channels=63 cov_cols=64
Rep 15: cells=50,000 channels=63 cov_cols=64
Rep 16: cells=50,000 channels=63 cov_cols=64
Rep 17: cells=50,000 channels=63 cov_cols=64
Rep 18: cells=50,000 channels=63 cov_cols=64
Rep 19: cells=50,000 channels=63 cov_cols=64


In [9]:
### generate the dictionary script for looping
# Manifest for cell subsamples (2 columns: h5ad, cov)
with open("Data/sc_annot_files_subsets/helper_files/bm_cell_subsample_manifest.txt", "w") as f:
    for i_rep in range(n_reps):
        h5ad = f"/data/cteu/2_users/zyu_lab/linke/chip_gwas_rev/{out_path}/bm_standard.ncell_50k_rep{i_rep}.h5ad"
        cov  = f"/data/cteu/2_users/zyu_lab/linke/chip_gwas_rev/{cov_path}/bm_standard.ncell_50k_rep{i_rep}.cov"
        f.write(f"{h5ad}\t{cov}\tanno\n")

print("Manifests written.")

Manifests written.


## munged GS 500 genes subsampling

In [ ]:
### generate manifest file for subgene analysis
gene_manifest = "Data/sc_annot_files_subsets/helper_files/onek1k_gene_subsample_manifest.txt"
with open(gene_manifest, "w") as f:
    h5ad = "Data/sc_annot_files/onek1k_sc_filtered.h5ad"
    cov  = "Data/sc_annot_files_qc/onek1k_sc_covs.tsv"
    f.write(f"{h5ad}\t{cov}\tpredicted.celltype.l2\n")
print(f"Gene subsample manifest written → {gene_manifest}")

In [10]:
### generate manifest file for subgene analysis
gene_manifest = "Data/sc_annot_files_subsets/helper_files/bm_gene_subsample_manifest.txt"
with open(gene_manifest, "w") as f:
    h5ad = "Data/sc_annot_files/BL_standard_design.h5ad"
    cov  = "Data/sc_annot_files_qc/BM_standard_design_covs.tsv"
    f.write(f"{h5ad}\t{cov}\tpredicted.celltype.l2\n")
print(f"Gene subsample manifest written → {gene_manifest}")

Gene subsample manifest written → Data/sc_annot_files_subsets/helper_files/bm_gene_subsample_manifest.txt
